# Cartography of Memory: Constructing the Exhibit

The [Cartography of Memory](https://matthewkollmer.com/cartography-of-memory/) sprung out of my reading of Mikhail Bakhtin's _The Dialogic Imagination_. As I read his essay on narrative time and space, I found myself thinking about what his notion of the __chronotope__ meant in terms of its relationship to place formation. Bakhtin defines chronotopes as "the intrinsic connectedness of temporal and spatial relationships that are artistically expressed in literature" (pg. 84). They are, in other words, how time and space acquire meaning together through narrative. I was interested in how this idea might also explain the places we carry in memory: particular locations made meaningful by encounters, experiences, and the stories we tell about them. I posited that these memories were constitutive of places and our personal narratives. They marked locations and time periods, or moments, as meaningful, underscoring that intrinsic connectedness that Bakhtin called chronotopes. Understood this way, I began to think of memories and the places they formed and held within them as chronotopes––units of measure wherein time and space were bound together. And understood as units of measure, I realized I could start collecting chronotopes, and that I could organize them as data.

So, I began collecting them. I located remembered places in Google maps, recorded their coordinates, and wrote descriptions of the memories I associated with them. At first, these descriptions were mostly commentary. But as I assembled and revised them, a larger narrative began to emerge. It was becoming an experimental memoir. The chronotopes, as Bakhtin observed, were unfolding narrative time. This experimental document was composed of chronotopes as data and a digital interface as the format. Through lots of revisions and testing, I included functionalities that let readers follow the chronotopes chronologically, explore them by themes, and select groups to read together. In the end, I had the Cartography of Memory, a digital narrative exhibit that gave readers a real sense of the places wherein my life unfolded––a chronotopic view of my memories through a digital mapping interface.

I wanted to make this project replicable for anyone interested in creative digital mapping and digital narrative. This notebook and the accompanying files break down the process so you can build your own Cartography of Memory. It begins with the collection of chronotopes, though, so you'll first need to write your own `chronotopes.csv`. This is simultaneously a data curation task, a creative writing task, and a personal reflection. Here's how to do it:

1) Copy the [`chronotopes.csv` file](https://raw.githubusercontent.com/MatthewKollmer/cartography-of-memory/main/chronotopes.csv). Delete my rows, but save the _necessary_ columns: `Location`, `Description`, `Latitude`, `Longitude`, `Year`, `Chronology`, and `Sentiment`. These columns must be in place for this whole process to work programmatically. 
2) You may save the other columns, too––the "theme" columns––but they aren't strictly necessary. In fact, you can even replace them with your own theme columns, should you choose. Any themes you include, though, must have `yes` or `no` data values. This data should tell you if the given chronotope is associated with the given theme, or not.
3) Once you've prepared your template of the `chronotopes.csv`, you're ready to start remembering. What places were meaningful, significant, or narratively consequential to your life? Give each place a row in the data, filling in the necessary column values as follows:
    - `Location`: the name of the place.
    - `Description`: a narrative textual description of the place. You can think of these descriptions as remembrances, creative non-fiction, or factual/historical notes. It's up to you. They should be no more than about 400 words, though, so they're readable in the exhibit.
    - `Latitude`: the latitude of the place. You can determine latitude using Google maps or other GPS software.
    `Longitude`: the longitude of the place. You can determine longitude using Google maps or other GPS software.
    - `Year`: the year in which you associate your memories of the place.
    - `Chronology`: the sequence number of each chronotope (i.e., the order in which you want them to be read in the exhibit). This must be written as a numeral (1, 2, 3, etc.) and they must follow a strict sequence.
    `Sentiment`: this column will house the valence scores, calculated through the code below. As you curate your chronotopes, you may leave this blank.
4) If you use the theme columns or write your own, you'll need to add `yes` or `no` labels. `yes` means the given chronotope should be associated with the given theme. `no` means the given chronotope shouldn't be associated with the given theme. All chronotopes must have `yes` or `no` values for each theme.

The number of chronotopes you draft is entirely up to you, but be sure to carve out time for this part of the process. My first iteration of this project has 229 chronotopes. It took me over three years and lots of revisions to compile them as they are. Remember: writing `chronotopes.csv` is a dynamic data curation exercise. It's also a creative exercise, a reflection on place and memory that can go on as long as you like. But if it's going to be as substantial as the example of this project I've constructed, it will take a long time to put together.

Finally, throughout this walkthrough, I'm assuming some familiarity with Python and HTML, particularly if you want to change how the exhibit works. That knowledge isn't strictly necessary to follow along or recreate the exhibit with your own data, though. The settings and prose you need to replace are identified below. But if you're not a progammer, no worries. Just follow the data curation steps exactly as described above and double-check that your URL pathways have been changed at step no. 3. Then run the cells in this notebook in order. By the end, you'll have an HTML version of your own Cartography of Memory.

## 1. Set Up Python Environment

I've created an `environment.yml` that specifies the Python environment for this walkthrough. This step isn't strictly necessary either; altough, whatever environment you run should have compatible versions of NumPy, PyArrow, Datasets, PyTorch, and Transformers. But if you want to construct the environment I used to build Cartography of Memory, make sure you have Conda installed then run these lines in the Terminal:

```bash
conda env create -f environment.yml
conda activate cartography-of-memory
```

## 2. Import Libraries

To work with and apply the data from `chronotopes.csv`, we'll be working with the following libraries:

In [1]:
import csv
import json
from urllib.request import urlopen
from io import StringIO

import os
import re
import tempfile

## 3. Point the Notebook to Your Collection

First, fork this repository and replace `chronotopes.csv` with your own collection. Keep it beside this notebook. Then the code below will allow you to load `chronotopes.csv` to complete the rest of the steps.

You'll also notice the exhibit template file: `exhibit_template.html`. This is a pre-formatted and designed version of the exhibit that becomes compatible with new versions of `chronotopes.csv`. If you're familiar with CSS and HTML, you may also want to make small edits to the exhibit design, but be careful: you can change colors, fonts, spacing, and visible wording, but keep the four `__LIFEMAP_...__` placeholders, the `cartography-of-memory` ID, and the existing control IDs and classes. The notebook and JavaScript use these names to connect your data to the exhibit's controls. Also, keep CSS rules scoped to `.cartography-exhibit` so design changes stay within the exhibit, not leak out into other parts of webpages that this exhibit might become part of.

Lastly, the code below loads the Basemap from OpenFreeMap. This is a changeable variable, too, but again, be careful: use a [MapLibre-compatible style JSON URL](https://maplibre.org/maplibre-gl-js/docs/API/type-aliases/MapOptions/#style), not a map webpage or an individual tile URL. Its map resources must be accessible to the browser, including any required provider key. The 3D buildings come from the style's building layers, too, so be sure to adjust the camera settings and 3D/See up close controls if your replacement has no buildings. Also update the template's OpenFreeMap attribution for your chosen provider and retain its required credits.

In [2]:
CSV_FILE = "chronotopes.csv"

TEMPLATE_FILE = "exhibit_template.html"
OUTPUT_DIR = "generated"
BASEMAP_STYLE = "https://tiles.openfreemap.org/styles/liberty"

# Keep these seven column names when adapting the CSV
REQUIRED_COLUMNS = ["Location", "Description", "Latitude", "Longitude","Year", "Sentiment", "Chronology"]

## 4. Read the CSV and Identify its Themes

Now we can read `chronotopes.csv` using `DictReader`. This keeps each description together, including quoted commas and paragraph breaks, in case your chronotope descriptions have those textual features. And we'll need to read your theme columns separately from the `REQUIRED_COLUMNS`. Remember: the theme columns can vary just as long as each chronotope theme value reads as either `yes` or `no`.

In [3]:
with open(CSV_FILE, encoding="utf-8-sig", newline="") as source:
    csv_text = source.read()

reader = csv.DictReader(StringIO(csv_text, newline=""))
headers = reader.fieldnames
raw_rows = list(reader)

THEMES = []
for column in headers:
    if column not in REQUIRED_COLUMNS:
        THEMES.append(column)
print(f"Loaded {len(raw_rows)} chronotopes.")

Loaded 228 chronotopes.


## 5. Estimate Sentiment for the Heatmap

The themes give readers one way to compare the places constructed through `chronotopes.csv`, but I also wanted to make their emotional associations were visible. This led to the idea of adding a sentiment heatmap. To build it, I needed a value for each description that runs from negative to positive. This dimension of sentiment is called __valence__. The steps below train a model to estimate valence scores, then put the results in the `Sentiment` column that the map can read and use for the sentiment layer. For example, a score of `0.7` means a relatively positive valence. The map would render the space around that point with darker purple/blue. A score of `0.3`, on the other hand, would mean a negative valence, and the map would render the space around it with a lighter yellow/green. This color palette, by the way, is called __Viridis__. I chose it because it is color-blind friendly. It's possible to change it, though. To do so, you just need to find `const sentimentColors` in `exhibit_template.html` and replace its 256 hexadecimal colors with another 256-color palette, ordered from the lowest score to the highest. Both the heatmap and its legend use this array, so rebuilding the exhibit would apply your new palette to both.

To get the valence scores for the sentiment heatmap, I fine-tuned a BERT model using two annotated collections: [EmoBank](https://github.com/JULIELab/EmoBank/blob/master/corpus/README.md) and [Fiction4Sentiment](https://huggingface.co/datasets/chcaa/fiction4sentiment). EmoBank includes several kinds of English writing, including literary texts, with valence ratings on a 1–5 scale. Fiction4Sentiment contributes English prose and poetry with human ratings on a 0–10 scale. These became my human-annotated training data for BERT. Although they aren't _perfectly_ aligned with my chronotope textual descriptions, I figured they'd work well enough. In my case, I drafted the chronotope descriptions as literary texts––brief, creative non-fiction. These training data sources are built from literary texts, so the sentiment labels will be more compatible than other forms of text (i.e., journalism, textbook writing, science writing, etc.). 

You're welcome to substitute other training data, but remember: it must pair text with meaningful valence annotations that you can bring onto the same scale. In this case, I took the 1-5 scale labels from EmoBank and the 1-10 scale labels from Fiction4Sentiment and converted them to a 0 to 1 scale. Also, remember that the choice of training material matters. A collection of movie reviews, for example, would teach BERT from a different kind of writing than the prose and poetry included in my approach.

Once trained, BERT scores each sentence in a description. Then all the scores per sentence in the description are averaged, creating an overall sentiment score per description. I also gave sentences farther from 0.5 greater weight when combining them into the single score. This was an interpretive choice made through trial and error. In the first iterations of this sentiment process, I found that a simple average let descriptive sentences, which were often more neutral, dilute the more polarized passages and brought many memories close to the middle of the scale. To account for those ubiquitious descriptive and neutral sentences, I decided to weigh the more extreme sentiment sentences more heavily so their descriptions would show greater variance in the heatmap.

### A Note on BERT Training

Please account for the time it takes to fine-tune BERT. Also, on macOS, the pinned PyTorch 2.6 release requires Apple Silicon. I'd avoid running this step on an Intel Mac. Also, double-check whether you have a supported GPU: PyTorch can use CUDA on a compatible NVIDIA setup, or MPS on supported Apple Silicon systems. Otherwise, training runs on the CPU. CPU training can take hours, so best avoid it, if possible.

Last thing: if training isn't possible on your computer, you could technically assign the scores yourself. To do this, you'd have to fill the `Sentiment` column with values between 0 and 1, reflecting your estimated negative or positive associations with each place and memory. I considered doing this, too, but ultimately felt that I'd learn more if I went with a computational approach which let me reflect on my sentiment towards these chronotopes as expressed in my own writing.

In [4]:
# Set the backend before importing Transformers.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import time
import nltk
import numpy as np
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)
MODEL_NAME = "google-bert/bert-base-uncased"
MODEL_DIR = os.path.join(OUTPUT_DIR, "sentiment-model")
MAX_TOKENS = 256

/opt/anaconda3/envs/cartography-of-memory/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Compiling the Training Data

From EmoBank, we're using the `text` and `V` columns in `emobank.csv`, which combines reader and writer judgments of valence. The code below converts its 1–5 ratings to 0–1 by subtracting 1 and dividing by 4. It also keeps the collection's existing `train`, `dev`, and `test` groups for training, validation, and testing.

From Fiction4Sentiment, we're using the English entries (`org_lang == "en"`): Hemingway's prose and Plath's poetry. The `label` column contains the mean human rating, which we divide by 10 to convert its 0–10 scale to 0–1. The code uses those human labels and leaves out the additional machine-generated scores. It divides the English material into 80 percent training, 10 percent validation, and 10 percent testing, using a fixed seed (42, the secret of the universe!) so the division can be repeated, then combines the training and validation groups with EmoBank's corresponding groups.

To learn more about these datasets, see the [EmoBank documentation](https://github.com/JULIELab/EmoBank/blob/master/corpus/README.md), the [Fiction4Sentiment dataset card](https://huggingface.co/datasets/chcaa/fiction4sentiment), and the [paper describing its annotations](https://ceur-ws.org/Vol-3834/paper98.pdf).

In [5]:
emobank_url = (
    "https://raw.githubusercontent.com/JULIELab/EmoBank/master/"
    "corpus/emobank.csv"
)
with urlopen(emobank_url) as response:
    emobank_rows = list(csv.DictReader(StringIO(response.read().decode("utf-8"))))

emobank = {}
for split in ("train", "dev", "test"):
    emobank[split] = Dataset.from_list([
        {"text": row["text"], "labels": (float(row["V"]) - 1) / 4}
        for row in emobank_rows if row["split"] == split
    ])

fiction_source = load_dataset("chcaa/fiction4sentiment", split="train")
fiction = Dataset.from_list([
    {"text": row["text"], "labels": float(row["label"]) / 10}
    for row in fiction_source if row["org_lang"] == "en"
])
fiction_split = fiction.train_test_split(test_size=0.2, seed=42)
fiction_holdout = fiction_split["test"].train_test_split(test_size=0.5, seed=42)

training_data = concatenate_datasets([emobank["train"], fiction_split["train"]])
validation_data = concatenate_datasets([emobank["dev"], fiction_holdout["train"]])
test_data = {"EmoBank": emobank["test"], "Fiction4": fiction_holdout["test"]}
print(f"Training: {len(training_data):,}; validation: {len(validation_data):,}")
print("Test examples:", {name: len(data) for name, data in test_data.items()})

Training: 10,863; validation: 1,350
Test examples: {'EmoBank': 1000, 'Fiction4': 351}


### Prepare the Text for BERT

We also need to prepare this training text for BERT. To do that, the code below loads [BERT's tokenizer](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/bert) and converts the training, validation, and test texts into numerical inputs, keeping the human valence scores as labels. Each text is limited to `MAX_TOKENS`, set here to 256 tokens. Anything beyond that limit is truncated. The final lines load BERT with a new output layer configured to predict one numerical valence score, ready for the training step below.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_TOKENS)

training_tokens = training_data.map(tokenize, batched=True, remove_columns=["text"])
validation_tokens = validation_data.map(tokenize, batched=True, remove_columns=["text"])
test_tokens = {
    name: data.map(tokenize, batched=True, remove_columns=["text"])
    for name, data in test_data.items()
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

Map: 100%|██████████| 351/351 [00:00<00:00, 66225.85 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Train the Model

And below will show you how I opted to train the model. I used three epochs and watched the mean absolute error rate. Three epochs give the model three passes through the training examples, a manageable starting point for fine-tuning a model that has already learned language patterns. Mean absolute error, or MAE, is the average distance between its predicted scores and the human labels on our 0–1 scale. Lower values mean closer agreement.

In [7]:
def measure_error(prediction):
    scores = np.clip(prediction.predictions.reshape(-1), 0, 1)
    return {"mae": float(np.abs(scores - prediction.label_ids).mean())}

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)
print("Training device:", training_args.device)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_tokens,
    eval_dataset=validation_tokens,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=measure_error,
)
training_started = time.perf_counter()
trainer.train()
print(f"Training time: {(time.perf_counter() - training_started) / 60:.1f} minutes")
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)


Training device: mps


Epoch,Training Loss,Validation Loss,Mae
1,0.005000,0.003610,0.043533
2,0.002900,0.003286,0.041125
3,0.001800,0.003290,0.041486


Training time: 8.8 minutes


('generated/sentiment-model/tokenizer_config.json',
 'generated/sentiment-model/special_tokens_map.json',
 'generated/sentiment-model/vocab.txt',
 'generated/sentiment-model/added_tokens.json',
 'generated/sentiment-model/tokenizer.json')

### Double-Check the Model

Just to double-check, it's also good to see how the model performed against the human labels. To do that, I just compared its predictions with the annotations in the held-out EmoBank and Fiction4Sentiment test sets, calculating MAE separately for each. The code also calculates a baseline error using the mean training score as the prediction for every test example. A lower model MAE shows an improvement over that baseline, though these tests don't establish how well the model reads our own chronotopes.

In [8]:
baseline = float(np.mean(training_data["labels"]))
for name, data in test_tokens.items():
    result = trainer.predict(data)
    scores = np.clip(result.predictions.reshape(-1), 0, 1)
    model_mae = float(np.abs(scores - result.label_ids).mean())
    baseline_mae = float(np.abs(baseline - result.label_ids).mean())
    print(f"{name}: model MAE {model_mae:.3f} - baseline MAE {baseline_mae:.3f}")

EmoBank: model MAE 0.040 - baseline MAE 0.062


Fiction4: model MAE 0.043 - baseline MAE 0.064


### Deploy the Fine-Tuned Model on Our Chronotopes

We can now apply the model to our chronotopes. The code uses [NLTK's sentence tokenizer](https://www.nltk.org/api/nltk.tokenize.html) to split each `Description` into sentences, prepares them with BERT's tokenizer, and predicts their valence scores, clipping the results to 0–1. It then combines each description's sentence scores into a weighted average, giving scores farther from 0.5 more influence.

These results update `Sentiment` in the loaded records and in the original local `chronotopes.csv`. Before saving, the code checks that the CSV hasn't changed since step 4, then writes a temporary copy and replaces the original when the write is complete. The exhibit below uses these updated records, while `all_sentences` and `sentence_scores` remain available if you want to inspect individual predictions.

In [9]:
def weighted_valence(scores):
    """Give sentence scores farther from 0.5 more influence."""
    scores = np.asarray(scores, dtype=float)
    if not scores.size:
        return None
    weights = np.abs(scores - 0.5)
    if weights.sum() == 0:
        return 0.5
    return float(np.average(scores, weights=weights))

nltk.download("punkt_tab", quiet=True)
sentence_groups = [
    nltk.sent_tokenize((row.get("Description") or "").strip())
    for row in raw_rows
]
all_sentences = [sentence for group in sentence_groups for sentence in group]

if all_sentences:
    description_data = Dataset.from_dict({"text": all_sentences})
    description_tokens = description_data.map(
        tokenize, batched=True, remove_columns=["text"]
    )
    sentence_scores = np.clip(
        trainer.predict(description_tokens).predictions.reshape(-1), 0, 1
    )
else:
    sentence_scores = np.array([])

offset = 0
for row, sentences in zip(raw_rows, sentence_groups):
    scores = sentence_scores[offset:offset + len(sentences)]
    sentiment = weighted_valence(scores)
    row["Sentiment"] = "" if sentiment is None else f"{sentiment:.4f}"
    offset += len(sentences)

with open(CSV_FILE, encoding="utf-8-sig", newline="") as source:
    if source.read() != csv_text:
        raise ValueError(
            f"{CSV_FILE} changed after it was loaded. Rerun step 4 and this scoring "
            "cell before saving, so your edits are preserved."
        )

csv_folder = os.path.dirname(os.path.abspath(CSV_FILE))
temporary_csv = None
try:
    with tempfile.NamedTemporaryFile(
        mode="w", encoding="utf-8", newline="", dir=csv_folder,
        prefix=".chronotopes-", suffix=".tmp", delete=False,
    ) as output:
        temporary_csv = output.name
        writer = csv.DictWriter(output, fieldnames=headers)
        writer.writeheader()
        writer.writerows(raw_rows)
    os.replace(temporary_csv, CSV_FILE)
finally:
    if temporary_csv and os.path.exists(temporary_csv):
        os.remove(temporary_csv)

with open(CSV_FILE, encoding="utf-8-sig", newline="") as source:
    csv_text = source.read()
print("Updated Sentiment column:", CSV_FILE)
for row in raw_rows[:5]:
    print(row["Location"], row["Sentiment"])

Map: 100%|██████████| 2094/2094 [00:00<00:00, 69379.37 examples/s]


Updated Sentiment column: chronotopes.csv
St. Paul, MN 0.5595
Afton, MN 0.5053
Duluth, MN 0.5168
Selma's Ice Cream Parlor, Afton, MN 0.5933
Grand Rapids, MN 0.5807


## 6. Configure Side Panel Information

While the textual descriptions provide information about individual places and memories, I also wanted to ensure users could quickly understand their selected groupings or visual layers. To do that, I added the side panel to the exhibit. To get it to work functionally in your version, you'll need to edit the code below.

I've separated the side panel labels into two groups. `DISPLAY_NARRATIVES` uses the interface's fixed keys: `all`, `sentiment`, and `chronology`. Keep those keys if you want labels for those views. `THEME_NARRATIVES` contains replaceable examples corresponding to my categories. Change them to match your own theme columns, and add a key such as `"Education:no"` if you want separate commentary for that theme's No selection.

All of this commentary, however, is optional. A theme still works if you leave its commentary empty or omit its narrative key, but if you want labels to appear in the side bar based on user selections, you should edit the code below.

In [10]:
# Fixed display keys. Add your prose between the quotation marks.
DISPLAY_NARRATIVES = {
    "all": "All chronotopes are mapped and visible.",
    "sentiment": "Positive or negative sentiments are rendered in color.",
    "chronology": "Chronological lines are displayed, adding directionality.",
}

# Replace these examples with your own theme headers; none is required.
THEME_NARRATIVES = {
    "Lived": "These are places I've lived.",
    "Visited": "These are places I've visited.",
    "Childhood": "These are places I associate with my childhood.",
    "Adolescence": "These are places from my adolescence.",
    "Adulthood": "These are places from my adulthood.",
    "Work": "These are places where I've worked.",
    "Recreation": "These are places I associate with recreation.",
    "Culture": "These are places where I experienced or engaged with culture.",
    "Education": "These are places related to my education.",
    "Food": "These are places I associate with food.",
    "Hard Lessons": "These are places where I've learned hard lessons.",
    "Sacred Sites": "These are places considered sacred.",
    # "Education:no": "Commentary on memories outside this theme.",
}

NARRATIVES = {
    "layers": {**DISPLAY_NARRATIVES, **THEME_NARRATIVES},
}

## 7. Turn the Rows into Map Records

At this point, we have the collection and any commentary we want to include, but we still need to convert the CSV's numeric text (lat/long, chronology, etc.) into numbers the map can use. The loop below does that and gathers each row into the record structure expected by the template. Blank years and sentiment scores become `None`, which will become `null` when we pass the records to JavaScript.

Each record also receives an `id` for selection and an `order` value recording its original row position. The browser sorts by `chronology`, using `order` to break ties. This was just a little failsafe after realizing that it's quite easy to make mistakes when writing the chronology order.

Finally, the yes/no values go into `themeValues`. Keeping them together lets the template use your chosen column names without confusing a theme with an internal field. This is the last transformation of the collection before we assemble the page.

In [11]:
places = []
for index, row in enumerate(raw_rows):
    place = {
        "id": f"place-{index + 1:03d}",
        "order": index,
        "location": row["Location"],
        "description": row["Description"],
        "latitude": float(row["Latitude"]),
        "longitude": float(row["Longitude"]),
        "year": int(row["Year"]) if row["Year"].strip() else None,
        "sentiment": float(row["Sentiment"]) if row["Sentiment"].strip() else None,
        "chronology": float(row["Chronology"]),
        "themeValues": {},
    }
    for theme in THEMES:
        place["themeValues"][theme] = row[theme].strip().lower()
    places.append(place)

print(f"Prepared {len(places)} memories for the exhibit.")


Prepared 228 memories for the exhibit.


## 8. Prepare the Records and Prose for the Browser

We now need to put our collection into a format the exhibit can read. The code below converts the map records, theme names, side panel commentary, and basemap setting into JSON. This is a text format that lets us pass the information from Python to the JavaScript that runs the exhibit.

The `embedded_json()` function also converts characters such as `<`, `>`, and `&` into codes that won't interfere with the HTML. Finally, `payloads` pairs each piece of information with its placeholder name in the template, ready for the next step.

In [12]:
def embedded_json(value):
    serialized = json.dumps(value, ensure_ascii=False, allow_nan=False)
    return (
        serialized.replace("&", "\\u0026")
        .replace("<", "\\u003c")
        .replace(">", "\\u003e")
    )

payloads = {
    "__LIFEMAP_BASEMAP__": embedded_json(BASEMAP_STYLE),
    "__LIFEMAP_DATA__": embedded_json(places),
    "__LIFEMAP_THEMES__": embedded_json(THEMES),
    "__LIFEMAP_NARRATIVES__": embedded_json(NARRATIVES),
}


## 9. Combine the Collection with the Template

Now we can add our collection to `exhibit_template.html`. The template already contains the layout and code for the map, dropdown menus, popups, and other controls. It also contains four placeholder names beginning with `__LIFEMAP_`, marking where our data and settings belong. The code below reads the template and replaces those placeholders with the JSON we prepared in step 8.

The result is stored in `exhibit`, which now contains the HTML for the new version of Cartography of Memory.

In [13]:
with open(TEMPLATE_FILE, encoding="utf-8") as source:
    template = source.read()

exhibit = re.sub(
    r"__LIFEMAP_(?:DATA|THEMES|NARRATIVES|BASEMAP)__",
    lambda match: payloads[match.group()],
    template,
)

## 10. Save the exhibit

Now we can write the exhibit to `generated/exhibit.html`. Rerunning this cell replaces that file, so be sure everything is configured how you prefer before running the cell below.

In [14]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_file = os.path.join(OUTPUT_DIR, "exhibit.html")
with open(output_file, "w", encoding="utf-8") as destination:
    destination.write(exhibit)
print("Created:", output_file)

Created: generated/exhibit.html


And that's a wrap! Now you can take your `exhibit.html` and host it online as a standalone webpage or as part of a longer HTML page.